# 03 — KPIs marketing & stratégie segment × canal

**Modules couverts : M5 (KPIs des campagnes) et M7 (stratégie de ciblage par canal)**  
Responsable : Pascal

Ce notebook calcule les KPI des campagnes marketing, puis construit une stratégie
segment × canal à partir du dataset officiel de segmentation clients.

**Entrées :**
- `data/raw/marketing_data.csv` : campagnes marketing ;
- `data/raw/sales_data.csv` : ventes utilisées pour calculer l'AOV ;
- `data/processed/dataset_clean.csv` : données clients/ventes nettoyées ;
- `segments_clients*.csv` : dataset officiel de segmentation, recherché sous la racine du projet.

**Sorties :** `data/processed/marketing_kpis.csv` et `data/processed/strategie_segments.csv`.

**Hypothèses et limites :**
1. Les campagnes ne contiennent pas de revenu réel par campagne. Le revenu estimé est donc `Conversions × AOV`, où l'AOV est calculé depuis `sales_data.csv`.
2. Les ventes actuelles contiennent cinq lignes et cinq `Sale_ID` uniques. L'AOV est calculé comme le revenu total des lignes (`Quantity × Sale_Price`) divisé par le nombre de ventes.
3. Le dataset officiel détecté contient une colonne `cluster` avec les valeurs `0`, `1`, `2` et `3`, sans profil métier associé. Ces clusters sont conservés tels quels ; aucune correspondance fictive avec Premium, Standard ou Economique n'est créée.
4. Les campagnes ne fournissent pas de KPI par segment client. Pour un cluster sans règle métier nommée, le canal recommandé repose donc sur le meilleur ROI global, calculé dynamiquement.
5. Les divisions par zéro produisent une valeur manquante contrôlée, jamais une valeur infinie.

In [12]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

## 0. Localisation des dossiers `data/raw` et `data/processed`

Fonctionne que le notebook soit lancé depuis `notebooks/` (cas normal) ou depuis la racine du projet.

In [33]:
def trouver_base_dir() -> Path:
    origines = []
    if "__file__" in globals():
        origines.append(Path(__file__).resolve().parent)
    origines.append(Path.cwd().resolve())

    for origine in origines:
        candidats = [origine, *origine.parents]
        for candidat in candidats:
            if (candidat / "data" / "raw" / "marketing_data.csv").exists():
                return candidat
            projets_enfants = candidat.glob("*/data/raw/marketing_data.csv")
            for fichier_marketing in projets_enfants:
                return fichier_marketing.parents[2]

    raise FileNotFoundError(
        "Impossible de trouver la racine du projet contenant data/raw/marketing_data.csv."
    )


BASE_DIR = trouver_base_dir()
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Racine du projet       : {BASE_DIR}")
print(f"Dossier de donnees     : {DATA_RAW}")
print(f"Dossier de sortie      : {DATA_PROCESSED}")

Racine du projet       : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA
Dossier de donnees     : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA\data\raw
Dossier de sortie      : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA\data\processed


## 1. Chargement et validation des données

`dataset_clean.csv` est cherché dans `data/processed/` (sortie M2), avec repli sur `data/raw/` s'il n'y est pas encore.

In [34]:
COLONNES_MARKETING_ATTENDUES = {
    "Campaign_ID", "Channel", "Budget", "Impressions", "Clicks", "Conversions"
}
COLONNES_SALES_ATTENDUES = {"Sale_ID", "Quantity", "Sale_Price"}
COLONNES_DATASET_CLEAN_ATTENDUES = {"Customer_ID", "Quantity", "Sale_Price", "Total_Spent"}


def valider_colonnes(df: pd.DataFrame, attendues: set[str], nom_fichier: str) -> None:
    manquantes = sorted(attendues - set(df.columns))
    if manquantes:
        raise ValueError(
            f"{nom_fichier} : colonnes manquantes {manquantes}.\n"
            f"Colonnes disponibles : {list(df.columns)}"
        )
    if df.empty:
        raise ValueError(f"{nom_fichier} est vide.")


def charger_csv(chemin: Path, colonnes_attendues: set[str]) -> pd.DataFrame:
    if not chemin.exists():
        raise FileNotFoundError(f"Fichier introuvable : {chemin}")
    df = pd.read_csv(chemin)
    valider_colonnes(df, colonnes_attendues, chemin.name)
    return df


marketing = charger_csv(DATA_RAW / "marketing_data.csv", COLONNES_MARKETING_ATTENDUES)
sales = charger_csv(DATA_RAW / "sales_data.csv", COLONNES_SALES_ATTENDUES)
dataset_clean = charger_csv(
    DATA_PROCESSED / "dataset_clean.csv",
    COLONNES_DATASET_CLEAN_ATTENDUES,
)

print(f"Campagnes marketing : {len(marketing)}")
print(f"Ventes              : {len(sales)}")
print(
    f"Lignes dataset_clean : {len(dataset_clean)} | "
    f"Clients uniques : {dataset_clean['Customer_ID'].nunique()}"
)
dataset_clean.head()

Campagnes marketing : 5
Ventes              : 5
Lignes dataset_clean : 5 | Clients uniques : 4


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel,Product_Name,Category,Price,Brand,Name,Age,Gender,Location,Join_Date,Total_Spent,Revenue
0,1,101,2001,2023-01-15,2,50.0,Online,T-shirt,Clothing,25.0,Brand A,Alice,28,Female,New York,2022-05-10,500.0,100.0
1,2,102,2002,2023-01-16,1,75.0,In-Store,Jeans,Clothing,75.0,Brand B,Bob,35,Male,Los Angeles,2022-06-15,750.0,75.0
2,3,103,2001,2023-01-17,3,30.0,Online,Sneakers,Footwear,30.0,Brand C,Alice,28,Female,New York,2022-05-10,500.0,90.0
3,4,104,2003,2023-01-18,1,120.0,In-Store,Jacket,Outerwear,120.0,Brand D,Charlie,22,Male,Chicago,2022-07-20,300.0,120.0
4,5,105,2004,2023-01-19,2,45.0,Online,Hat,Accessories,22.5,Brand E,Diana,30,Female,Houston,2022-08-25,600.0,90.0


## 2. Partie M5 — KPIs par campagne

CTR, taux de conversion, CPC, CPA calculés directement sur `marketing_data.csv`. Le revenu et le ROI sont **estimés** via le panier moyen observé dans `dataset_clean.csv` (voir hypothèse 1 en introduction).

In [35]:
COLONNES_NUMERIQUES_MARKETING = [
    "Budget", "Impressions", "Clicks", "Conversions"
]


def verifier_colonnes_numeriques(df: pd.DataFrame, colonnes: list[str], nom_fichier: str) -> None:
    valeurs_invalides = {}
    for colonne in colonnes:
        valeurs = pd.to_numeric(df[colonne], errors="coerce")
        if valeurs.isna().any():
            valeurs_invalides[colonne] = int(valeurs.isna().sum())
    if valeurs_invalides:
        raise ValueError(
            f"{nom_fichier} contient des valeurs numeriques invalides : {valeurs_invalides}"
        )


def calculer_panier_moyen(sales: pd.DataFrame) -> float:
    verifier_colonnes_numeriques(sales, ["Quantity", "Sale_Price"], "sales_data.csv")
    ventes = sales["Quantity"] * sales["Sale_Price"]
    nombre_ventes = sales["Sale_ID"].nunique()
    if nombre_ventes == 0:
        raise ValueError("sales_data.csv ne contient aucune vente exploitable.")
    return float(ventes.sum() / nombre_ventes)


def division_sure(numerateur: pd.Series, denominateur: pd.Series) -> pd.Series:
    denominateur_non_nul = denominateur.where(denominateur != 0)
    return numerateur.div(denominateur_non_nul)


def calculer_kpis_campagnes(marketing: pd.DataFrame, aov: float) -> pd.DataFrame:
    verifier_colonnes_numeriques(
        marketing,
        COLONNES_NUMERIQUES_MARKETING,
        "marketing_data.csv",
    )
    resultats = marketing.copy()

    resultats["CTR (%)"] = (
        division_sure(resultats["Clicks"], resultats["Impressions"]) * 100
    ).round(2)
    resultats["Taux_Conversion (%)"] = (
        division_sure(resultats["Conversions"], resultats["Clicks"]) * 100
    ).round(2)
    resultats["CPC (EUR)"] = division_sure(
        resultats["Budget"], resultats["Clicks"]
    ).round(2)
    resultats["CPA (EUR)"] = division_sure(
        resultats["Budget"], resultats["Conversions"]
    ).round(2)
    resultats["Revenu_Estime (EUR)"] = (resultats["Conversions"] * aov).round(2)
    resultats["ROI (%)"] = (
        division_sure(
            resultats["Revenu_Estime (EUR)"] - resultats["Budget"],
            resultats["Budget"],
        )
        * 100
    ).round(1)

    colonnes_sortie = [
        "Campaign_ID", "Channel", "Budget", "Impressions", "Clicks",
        "Conversions", "CTR (%)", "Taux_Conversion (%)", "CPC (EUR)",
        "CPA (EUR)", "Revenu_Estime (EUR)", "ROI (%)",
    ]
    sortie = resultats[colonnes_sortie]
    colonnes_kpi = [colonne for colonne in colonnes_sortie if colonne.endswith("%)") or "EUR" in colonne]
    if sortie[colonnes_kpi].replace([float("inf"), float("-inf")], pd.NA).isna().all().any():
        raise ValueError("Un KPI est entierement indetermine : verifiez les denominateurs.")
    return sortie


aov = calculer_panier_moyen(sales)
print(f"Panier moyen estime (AOV) : {aov:.2f} EUR")

kpis = calculer_kpis_campagnes(marketing, aov)
kpis

Panier moyen estime (AOV) : 95.00 EUR


,Campaign_ID,Channel,Budget,Impressions,Clicks,Conversions,CTR (%),Taux_Conversion (%),CPC (EUR),CPA (EUR),Revenu_Estime (EUR),ROI (%)
0,1,Online,1000.0,50000,2000,150,4.00,7.50,0.50,6.67,14250.0,1325.0
1,2,In-Store,1500.0,30000,500,100,1.67,20.00,3.00,15.00,9500.0,533.3
2,3,Social,2000.0,40000,1500,200,3.75,13.33,1.33,10.00,19000.0,850.0
3,4,Email,500.0,20000,1000,50,5.00,5.00,0.50,10.00,4750.0,850.0
4,5,TV,3000.0,60000,3000,250,5.00,8.33,1.00,12.00,23750.0,691.7


In [38]:
kpis.to_csv(DATA_PROCESSED / "marketing_kpis.csv", index=False)
print(f"Ecrit : {(DATA_PROCESSED / 'marketing_kpis.csv').resolve()}")

Ecrit : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA\data\processed\marketing_kpis.csv


## 3. Partie M7 — Stratégie segment × canal

Le notebook recherche automatiquement le fichier `segments_clients*.csv` sous la racine
du projet. Il utilise la colonne `Segment` si elle existe, sinon normalise la colonne
réelle `cluster` ou `Cluster` en `Segment`, sans créer de segmentation provisoire.

Les clusters numériques du dataset actuel sont conservés tels quels. Comme aucun profil
métier n'est fourni pour ces clusters, la stratégie applique le meilleur ROI global comme
règle de repli et le signale dans la colonne `Profil`.

In [36]:
def trouver_dataset_segmentation(base_dir: Path) -> Path:
    candidats = sorted(
        chemin
        for chemin in base_dir.rglob("segments_clients*.csv")
        if chemin.is_file()
    )
    if not candidats:
        raise FileNotFoundError(
            f"Aucun dataset de segmentation segments_clients*.csv trouve sous {base_dir}."
        )
    if len(candidats) > 1:
        print(f"[!] Plusieurs datasets trouves, utilisation de : {candidats[0]}")
    return candidats[0]


def charger_segments_clients(base_dir: Path) -> pd.DataFrame:
    chemin = trouver_dataset_segmentation(base_dir)
    segments = pd.read_csv(chemin)
    valider_colonnes(segments, {"Customer_ID", "Total_Spent"}, chemin.name)

    if "Segment" in segments.columns:
        nom_segment = "Segment"
    elif "cluster" in segments.columns:
        nom_segment = "cluster"
    elif "Cluster" in segments.columns:
        nom_segment = "Cluster"
    else:
        raise ValueError(
            f"{chemin.name} : colonne de segmentation absente. "
            f"Colonnes disponibles : {list(segments.columns)}"
        )

    segments = segments.rename(columns={nom_segment: "Segment"})
    segments["Segment"] = segments["Segment"].astype(str)
    if segments["Segment"].str.strip().eq("").any():
        raise ValueError(f"{chemin.name} contient des segments vides.")

    print(f"[OK] Dataset de segmentation utilise : {chemin}")
    print(f"[OK] Segments reels : {sorted(segments['Segment'].unique())}")
    return segments


customers_segmentes = charger_segments_clients(BASE_DIR)
customers_segmentes

[OK] Dataset de segmentation utilise : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA\data\processed\segments_clients.csv
[OK] Segments reels : ['0', '1', '2', '3']


,Customer_ID,Name,Age,Gender,Location,Join_Date,Total_Spent,recency_days,purchase_frequency,monetary_total,avg_basket_value,Segment
0,2001,Ivy A.,50,Female,Philadelphia,2022-05-17,190.02,354,1,190.02,190.02,0
1,2002,Quinn B.,47,Female,Chicago,2023-08-02,1849.79,40,10,1849.79,184.98,2
2,2003,Sam C.,25,Female,San Diego,2023-04-06,1926.19,30,12,1926.19,160.52,3
3,2004,Quinn D.,24,Female,San Diego,2023-06-27,1259.16,39,8,1259.16,157.40,3
4,2005,Diana E.,56,Female,Houston,2023-08-09,413.48,164,3,413.48,137.83,0
...,...,...,...,...,...,...,...,...,...,...,...,...
395,2396,Diana F.,59,Female,New York,2023-05-28,2573.67,104,16,2573.67,160.85,2
396,2397,Rita G.,51,Male,Los Angeles,2023-04-27,1664.31,118,12,1664.31,138.69,2
397,2398,Quinn H.,29,Male,New York,2023-04-24,212.14,206,2,212.14,106.07,0
398,2399,Paul I.,69,Male,Los Angeles,2022-06-28,2064.78,46,12,2064.78,172.07,2


In [37]:
REGLES_SEGMENTS = {
    "Premium": ("ROI (%)", "max"),
    "Standard": ("CPA (EUR)", "min"),
    "Economique": ("Taux_Conversion (%)", "max"),
}

PROFILS_SEGMENTS = {
    "Premium": "Clients a forte valeur, cible prioritaire de fidelisation/upsell",
    "Standard": "Clients a valeur moyenne, a faire progresser vers Premium",
    "Economique": "Clients sensibles au prix, besoin d'un declencheur d'achat direct",
}

REGLE_CLUSTER_SANS_PROFIL = ("ROI (%)", "max")
PROFIL_CLUSTER_SANS_PROFIL = (
    "Cluster officiel issu de la segmentation clients "
    "(profil metier non fourni dans le dataset)"
)


def choisir_campagne(kpis: pd.DataFrame, metrique: str, mode: str) -> pd.Series:
    if metrique not in kpis.columns:
        raise ValueError(f"KPI requis absent : {metrique}")
    campagnes_valides = kpis.dropna(subset=[metrique])
    if campagnes_valides.empty:
        raise ValueError(f"Aucune valeur exploitable pour le KPI : {metrique}")
    indice = (
        campagnes_valides[metrique].idxmax()
        if mode == "max"
        else campagnes_valides[metrique].idxmin()
    )
    return campagnes_valides.loc[indice]


def construire_strategie_segments(
    customers_segmentes: pd.DataFrame,
    kpis: pd.DataFrame,
) -> pd.DataFrame:
    valider_colonnes(
        customers_segmentes,
        {"Customer_ID", "Total_Spent", "Segment"},
        "dataset de segmentation clients",
    )
    if kpis.empty:
        raise ValueError("Le tableau des KPI est vide.")

    resume = (
        customers_segmentes.groupby("Segment", dropna=False)
        .agg(
            Nb_Clients=("Customer_ID", "nunique"),
            Total_Spent_Moyen=("Total_Spent", "mean"),
        )
        .reset_index()
    )
    resume["Segment"] = resume["Segment"].astype(str)
    resume["Total_Spent_Moyen (EUR)"] = resume["Total_Spent_Moyen"].round(2)
    resume = resume.drop(columns="Total_Spent_Moyen")

    canaux, justifications, profils = [], [], []
    for segment in resume["Segment"]:
        metrique, mode = REGLES_SEGMENTS.get(segment, REGLE_CLUSTER_SANS_PROFIL)
        campagne = choisir_campagne(kpis, metrique, mode)
        qualificatif = "meilleur" if mode == "max" else "plus bas"

        canaux.append(campagne["Channel"])
        justifications.append(
            f"Canal avec le {qualificatif} {metrique} observe "
            f"({campagne[metrique]}) parmi les campagnes analysees"
        )
        profils.append(PROFILS_SEGMENTS.get(segment, PROFIL_CLUSTER_SANS_PROFIL))

    resume["Profil"] = profils
    resume["Canal_Recommande"] = canaux
    resume["Justification"] = justifications
    return resume[
        [
            "Segment",
            "Nb_Clients",
            "Total_Spent_Moyen (EUR)",
            "Profil",
            "Canal_Recommande",
            "Justification",
        ]
    ]


strategie = construire_strategie_segments(customers_segmentes, kpis)
strategie

,Segment,Nb_Clients,Total_Spent_Moyen (EUR),Profil,Canal_Recommande,Justification
0,0,114,353.55,Cluster officiel issu de la segmentation clien...,Online,Canal avec le meilleur ROI (%) observe (1325.0...
1,1,66,4131.06,Cluster officiel issu de la segmentation clien...,Online,Canal avec le meilleur ROI (%) observe (1325.0...
2,2,80,2556.17,Cluster officiel issu de la segmentation clien...,Online,Canal avec le meilleur ROI (%) observe (1325.0...
3,3,140,2138.29,Cluster officiel issu de la segmentation clien...,Online,Canal avec le meilleur ROI (%) observe (1325.0...


In [39]:
strategie.to_csv(DATA_PROCESSED / "strategie_segments.csv", index=False)
print(f"Ecrit : {(DATA_PROCESSED / 'strategie_segments.csv').resolve()}")

Ecrit : G:\Project\PlatformIO\SMD\Segmentation-Client-Marketing-IA\data\processed\strategie_segments.csv


In [40]:
marketing_sortie = pd.read_csv(DATA_PROCESSED / "marketing_kpis.csv")
strategie_sortie = pd.read_csv(DATA_PROCESSED / "strategie_segments.csv")

colonnes_kpi = [
    "CTR (%)", "Taux_Conversion (%)", "CPC (EUR)", "CPA (EUR)",
    "Revenu_Estime (EUR)", "ROI (%)",
]
assert len(marketing_sortie) == len(marketing)
assert not marketing_sortie[colonnes_kpi].isin([float("inf"), float("-inf")]).any().any()
assert set(strategie_sortie["Canal_Recommande"]).issubset(set(marketing["Channel"]))
assert strategie_sortie["Nb_Clients"].sum() == customers_segmentes["Customer_ID"].nunique()
assert set(strategie_sortie["Segment"].astype(str)) == set(customers_segmentes["Segment"].astype(str))

print("[OK] Validation des sorties terminee")
print(f"AOV : {aov:.2f} EUR")
print(f"Segments : {sorted(strategie_sortie['Segment'].astype(str).unique())}")
print(f"Canaux recommandes : {sorted(strategie_sortie['Canal_Recommande'].unique())}")
print("\nmarketing_kpis.csv")
print(marketing_sortie.to_string(index=False))
print("\nstrategie_segments.csv")
print(strategie_sortie.to_string(index=False))

[OK] Validation des sorties terminee
AOV : 95.00 EUR
Segments : ['0', '1', '2', '3']
Canaux recommandes : ['Online']

marketing_kpis.csv
 Campaign_ID  Channel  Budget  Impressions  Clicks  Conversions  CTR (%)  Taux_Conversion (%)  CPC (EUR)  CPA (EUR)  Revenu_Estime (EUR)  ROI (%)
           1   Online  1000.0        50000    2000          150     4.00                 7.50       0.50       6.67              14250.0   1325.0
           2 In-Store  1500.0        30000     500          100     1.67                20.00       3.00      15.00               9500.0    533.3
           3   Social  2000.0        40000    1500          200     3.75                13.33       1.33      10.00              19000.0    850.0
           4    Email   500.0        20000    1000           50     5.00                 5.00       0.50      10.00               4750.0    850.0
           5       TV  3000.0        60000    3000          250     5.00                 8.33       1.00      12.00              2375